# SupplyForge AI 2.0: KAN-36 - Calendar & Promotion Features



Tài liệu này thực hiện việc trích xuất các đặc trưng thời gian (Calendar Features) và cờ sự kiện khuyến mại (Promotion Features) từ cột `date` trong lưới Daily Grid.



Các đặc trưng lịch bao gồm: ngày trong tuần, ngày trong tháng, tuần trong năm, tháng, quý, ngày cuối tuần, ngày đầu tháng, ngày cuối tháng. Đặc trưng khuyến mại bao gồm cờ ngày lễ Việt Nam (is_holiday) và các chiến dịch e-commerce định kỳ như ngày đôi (Mega Sales), giữa tháng (Mid-Month) và ngày lương về (Salary Day) đi kèm thông tin tên chiến dịch, loại chiến dịch và cấp độ khuyến mãi.


### Quy tắc nghiệp vụ và Tiêu chuẩn Nghiệm thu (Acceptance Criteria):

1. **Bảo toàn dòng**: Kích thước lưới Daily SKU Grid không thay đổi (giữ nguyên 323,250 dòng).

2. **Chống rò rỉ dữ liệu**: Chỉ sử dụng lịch biểu biết trước tại ngày dự báo.

3. **Tính chính xác**: Gán nhãn đúng 100% các ngày nghỉ lễ Việt Nam và các đợt Mega Sales Shopee/Lazada.


In [1]:
import pandas as pd

import numpy as np

import os

import sys



# Liên kết thư mục scripts để import module

sys.path.append('/home/nguyen-dai-tung/supplyforge-ai/scripts')

from calendar_features import generate_calendar_features



DATA_DIR = '/home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH'

grid_path = os.path.join(DATA_DIR, 'calendar_daily_sales_metadata.csv')



print("Đang tải lưới dữ liệu Daily Calendar-SKU...")

df_grid = pd.read_csv(grid_path)

print(f"Kích thước lưới ban đầu: {df_grid.shape}")


Đang tải lưới dữ liệu Daily Calendar-SKU...


Kích thước lưới ban đầu: (323250, 43)


In [2]:
print("Đang chạy quy trình trích xuất đặc trưng lịch và chiến dịch khuyến mại...")

df_featured = generate_calendar_features(df_grid)

print(f"Kích thước lưới sau khi tính toán: {df_featured.shape}")

print("Danh sách đặc trưng lịch mới thêm vào lưới:")

print([col for col in df_featured.columns if col not in df_grid.columns])


Đang chạy quy trình trích xuất đặc trưng lịch và chiến dịch khuyến mại...


Đang trích xuất các đặc trưng lịch cơ bản...
Đang gán cờ ngày lễ Việt Nam (is_holiday)...
Đang gán cờ chiến dịch e-commerce định kỳ (campaign)...
Kích thước lưới sau khi tính toán: (323250, 56)
Danh sách đặc trưng lịch mới thêm vào lưới:
['day_of_week', 'day_of_month', 'week_of_year', 'month', 'quarter', 'is_weekend', 'is_month_start', 'is_month_end', 'is_holiday', 'campaign_name', 'campaign_type', 'promotion_level', 'is_sale_campaign']


In [3]:
print("=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===\n")



# 1. Xác thực số lượng dòng được bảo toàn

print(f"Số dòng trước: {len(df_grid)} | Số dòng sau: {len(df_featured)}")

assert len(df_grid) == len(df_featured), "Lỗi: Số lượng dòng lưới bị thay đổi!"

print("1. Đảm bảo bảo toàn số dòng lưới: Đạt\n")



# 2. Đảm bảo không chứa giá trị Null ở bất kỳ cột đặc trưng mới nào

new_cols = [col for col in df_featured.columns if col not in df_grid.columns]

for col in new_cols:

    null_count = df_featured[col].isnull().sum()

    print(f"Số lượng Null ở cột {col}: {null_count}")

    assert null_count == 0, f"Lỗi: Cột {col} chứa giá trị Null!"

print("2. Không chứa bất kỳ giá trị khuyết thiếu (Null) nào: Đạt\n")



# 3. Kiểm định gán nhãn cuối tuần (is_weekend)

weekend_check = df_featured[df_featured['day_of_week'].isin([5, 6])]['is_weekend'].all()

weekday_check = (df_featured[~df_featured['day_of_week'].isin([5, 6])]['is_weekend'] == 0).all()

assert weekend_check and weekday_check, "Lỗi: Gán cờ is_weekend không chính xác!"

print("3. Kiểm định gán nhãn cuối tuần (is_weekend): Đạt\n")



# 4. Kiểm định ngày lễ lớn (is_holiday) của Việt Nam

holiday_dates = ['2022-04-30', '2022-05-01', '2023-09-02']

for d in holiday_dates:

    df_d = df_featured[df_featured['date'] == d]

    if len(df_d) > 0:

        val = df_d['is_holiday'].iloc[0]

        print(f"Kiểm tra ngày lễ {d}: is_holiday={val}")

        assert val == 1, f"Lỗi: Ngày {d} không được nhận diện là ngày lễ!"

print("4. Kiểm định ngày lễ Việt Nam (is_holiday): Đạt\n")



# 5. Kiểm định chiến dịch e-commerce (is_sale_campaign)

campaign_dates = ['2022-11-11', '2022-12-12', '2022-05-15', '2022-08-25']

for d in campaign_dates:

    df_d = df_featured[df_featured['date'] == d]

    if len(df_d) > 0:

        val = df_d['is_sale_campaign'].iloc[0]

        name = df_d['campaign_name'].iloc[0]

        lvl = df_d['promotion_level'].iloc[0]

        print(f"Kiểm tra chiến dịch {d}: is_sale_campaign={val}, name={name}, level={lvl}")

        assert val == 1, f"Lỗi: Ngày {d} không được nhận diện là ngày sale!"

print("5. Kiểm định gán cờ chiến dịch e-commerce: Đạt\n")


=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===

Số dòng trước: 323250 | Số dòng sau: 323250
1. Đảm bảo bảo toàn số dòng lưới: Đạt

Số lượng Null ở cột day_of_week: 0
Số lượng Null ở cột day_of_month: 0
Số lượng Null ở cột week_of_year: 0
Số lượng Null ở cột month: 0
Số lượng Null ở cột quarter: 0
Số lượng Null ở cột is_weekend: 0
Số lượng Null ở cột is_month_start: 0
Số lượng Null ở cột is_month_end: 0
Số lượng Null ở cột is_holiday: 0
Số lượng Null ở cột campaign_name: 0
Số lượng Null ở cột campaign_type: 0
Số lượng Null ở cột promotion_level: 0
Số lượng Null ở cột is_sale_campaign: 0
2. Không chứa bất kỳ giá trị khuyết thiếu (Null) nào: Đạt



3. Kiểm định gán nhãn cuối tuần (is_weekend): Đạt

Kiểm tra ngày lễ 2022-04-30: is_holiday=1
Kiểm tra ngày lễ 2022-05-01: is_holiday=1
4. Kiểm định ngày lễ Việt Nam (is_holiday): Đạt

Kiểm tra chiến dịch 2022-11-11: is_sale_campaign=1, name=Mega Sale 11.11, level=2
Kiểm tra chiến dịch 2022-12-12: is_sale_campaign=1, name=Mega Sale 12.12, level=2
Kiểm tra chiến dịch 2022-05-15: is_sale_campaign=1, name=Mid-Month Sale, level=1
Kiểm tra chiến dịch 2022-08-25: is_sale_campaign=1, name=Salary Day Sale, level=1
5. Kiểm định gán cờ chiến dịch e-commerce: Đạt



In [4]:
print("=== LƯU ĐÈ LƯỚI THỜI GIAN ĐÃ ĐƯỢC LÀM GIÀU ĐẶC TRƯNG LỊCH ===")



# Bảo toàn các cột category để tránh mất định dạng phân loại của LightGBM

for col in ['brand_encoded', 'categoryId_encoded', 'warehouse_encoded']:

    df_featured[col] = df_featured[col].astype('category')



df_featured.to_csv(grid_path, index=False)

print(f"Đã lưu thành công lưới dữ liệu đặc trưng lịch tại: {grid_path}")


=== LƯU ĐÈ LƯỚI THỜI GIAN ĐÃ ĐƯỢC LÀM GIÀU ĐẶC TRƯNG LỊCH ===


Đã lưu thành công lưới dữ liệu đặc trưng lịch tại: /home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH/calendar_daily_sales_metadata.csv
